# 03 — In Season Feature Engineering

This notebook creates the weekly performance features used by the in season projection layer.

It does not modify the original preseason notebooks or their outputs.

The purpose of this notebook is to convert completed regular season games into team level offensive and defensive performance features that can later be blended with the frozen preseason model.

For a selected target week, the notebook:

- Loads only completed games from weeks before the target week
- Builds team level scoring and margin summaries
- Separates offense and defense
- Adds simple opponent adjusted performance features
- Applies early season shrinkage to reduce overreaction to small samples
- Saves a clean feature table for the weekly team strength notebook

This notebook is intentionally conservative after Week 1. One game should inform the model, not redefine it.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


## Paths and Weekly Settings

This notebook expects the outputs created by:

`02_Weekly_Data_Update.ipynb`

The only value that normally changes week to week is `TARGET_WEEK`.


In [2]:
PROJECT_ROOT = Path("../..")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
WEEKLY_DATA_DIR = PROCESSED_DIR / "weekly"

SEASON = 2026
TARGET_WEEK = 11
PRIOR_WEEK = TARGET_WEEK - 1

completed_games_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_completed_games.parquet"
)

prior_strength_path = (
    WEEKLY_DATA_DIR
    / f"week_{PRIOR_WEEK:02d}_sequential_team_strength.parquet"
)

print("Using completed games:", completed_games_path)
print("Using entering ratings:", prior_strength_path)

Using completed games: ..\..\data\processed\weekly\week_11_completed_games.parquet
Using entering ratings: ..\..\data\processed\weekly\week_10_sequential_team_strength.parquet


# Load Completed Games

Only games completed before the target week should exist in this file.

That leakage rule was enforced in `02_Weekly_Data_Update.ipynb`.


In [3]:
completed_games = pd.read_parquet(
    completed_games_path
)

completed_games = completed_games.sort_values(
    ["week", "gameday", "game_id"]
).reset_index(drop=True)

print(
    f"Completed games available before Week {TARGET_WEEK}:",
    len(completed_games)
)

display(
    completed_games[
        [
            "week",
            "gameday",
            "away_team",
            "away_score",
            "home_team",
            "home_score"
        ]
    ]
)


Completed games available before Week 11: 150


,week,gameday,away_team,away_score,home_team,home_score
0,1,2026-09-09,NE,10.0,SEA,13.0
1,1,2026-09-10,SF,27.0,LA,7.0
2,1,2026-09-13,ARI,26.0,LAC,14.0
3,1,2026-09-13,ATL,13.0,PIT,20.0
4,1,2026-09-13,BAL,41.0,IND,23.0
...,...,...,...,...,...,...
145,10,2026-11-15,NE,10.0,DET,35.0
146,10,2026-11-15,PIT,18.0,CIN,38.0
147,10,2026-11-15,SEA,21.0,LV,13.0
148,10,2026-11-15,SF,46.0,DAL,40.0


In [4]:
update_games = completed_games[
    completed_games["week"] == PRIOR_WEEK
].copy()

update_games = update_games.sort_values(
    ["gameday", "game_id"]
).reset_index(drop=True)

print(
    f"Week {PRIOR_WEEK} games used to update "
    f"Week {TARGET_WEEK} ratings:",
    len(update_games)
)

display(
    update_games[
        [
            "week",
            "gameday",
            "away_team",
            "away_score",
            "home_team",
            "home_score"
        ]
    ]
)

Week 10 games used to update Week 11 ratings: 14


,week,gameday,away_team,away_score,home_team,home_score
0,10,2026-11-12,WAS,24.0,NYG,21.0
1,10,2026-11-15,BUF,31.0,NYJ,12.0
2,10,2026-11-15,CAR,20.0,NO,17.0
3,10,2026-11-15,HOU,24.0,CLE,27.0
4,10,2026-11-15,JAX,40.0,TEN,23.0
5,10,2026-11-15,KC,35.0,ATL,13.0
6,10,2026-11-15,LA,34.0,ARI,6.0
7,10,2026-11-15,MIA,22.0,IND,24.0
8,10,2026-11-15,MIN,30.0,GB,10.0
9,10,2026-11-15,NE,10.0,DET,35.0


# Convert Prior Week Games to Team Game Rows

Only the most recently completed week is used for the new weekly update.

For Week 3:

- Week 2 team strength is the entering prior
- Week 2 game performance is the new evidence
- Week 1 is not counted again because it is already reflected in the Week 2 ratings

Each game becomes two rows:

- one from the home team's perspective
- one from the away team's perspective

In [5]:
home_rows = pd.DataFrame(
    {
        "game_id": update_games["game_id"],
        "week": update_games["week"],
        "team": update_games["home_team"],
        "opponent": update_games["away_team"],
        "is_home": 1,
        "points_for": update_games["home_score"],
        "points_against": update_games["away_score"]
    }
)

away_rows = pd.DataFrame(
    {
        "game_id": update_games["game_id"],
        "week": update_games["week"],
        "team": update_games["away_team"],
        "opponent": update_games["home_team"],
        "is_home": 0,
        "points_for": update_games["away_score"],
        "points_against": update_games["home_score"]
    }
)

team_games = pd.concat(
    [home_rows, away_rows],
    ignore_index=True
)

team_games["point_margin"] = (
    team_games["points_for"]
    - team_games["points_against"]
)

team_games = team_games.sort_values(
    ["team", "game_id"]
).reset_index(drop=True)

print("Team-game rows:", len(team_games))

display(team_games)

Team-game rows: 28


,game_id,week,team,opponent,is_home,points_for,points_against,point_margin
0,2026_10_LA_ARI,10,ARI,LA,1,6.0,34.0,-28.0
1,2026_10_KC_ATL,10,ATL,KC,1,13.0,35.0,-22.0
2,2026_10_LAC_BAL,10,BAL,LAC,1,20.0,23.0,-3.0
3,2026_10_BUF_NYJ,10,BUF,NYJ,0,31.0,12.0,19.0
4,2026_10_CAR_NO,10,CAR,NO,0,20.0,17.0,3.0
5,2026_10_PIT_CIN,10,CIN,PIT,1,38.0,18.0,20.0
6,2026_10_HOU_CLE,10,CLE,HOU,1,27.0,24.0,3.0
7,2026_10_SF_DAL,10,DAL,SF,1,40.0,46.0,-6.0
8,2026_10_NE_DET,10,DET,NE,1,35.0,10.0,25.0
9,2026_10_MIN_GB,10,GB,MIN,1,10.0,30.0,-20.0


# Attach Entering Team and Opponent Strength

Each team's performance is evaluated relative to what the model believed before the game was played.

For Week 3 projections, Week 2 ratings represent the information available entering Week 2.

This prevents the opponent adjustment from using future information or the same results that are currently being evaluated.

In [6]:
prior_strength = pd.read_parquet(
    prior_strength_path
).copy()

required_prior_columns = [
    "team",
    "weekly_team_strength",
    "weekly_strength_rank"
]

missing_prior_columns = [
    col
    for col in required_prior_columns
    if col not in prior_strength.columns
]

if missing_prior_columns:
    raise KeyError(
        "Missing prior-week strength columns: "
        + ", ".join(missing_prior_columns)
    )

prior_strength = prior_strength[
    required_prior_columns
].rename(
    columns={
        "weekly_team_strength": "entering_team_strength",
        "weekly_strength_rank": "entering_strength_rank"
    }
)

print(
    f"Week {PRIOR_WEEK} entering ratings loaded:",
    len(prior_strength)
)

display(
    prior_strength.sort_values(
        "entering_strength_rank"
    )
)

Week 10 entering ratings loaded: 32


,team,entering_team_strength,entering_strength_rank
0,BUF,7.483841,1
1,JAX,5.875962,2
2,SF,4.990056,3
3,LA,4.643452,4
4,SEA,4.207138,5
5,DEN,3.953856,6
6,KC,3.818115,7
7,DET,3.609975,8
8,CHI,3.472897,9
9,GB,3.446027,10


In [7]:
team_games = team_games.merge(
    prior_strength[
        [
            "team",
            "entering_team_strength",
            "entering_strength_rank"
        ]
    ],
    on="team",
    how="left",
    validate="many_to_one"
)

opponent_strength = prior_strength[
    [
        "team",
        "entering_team_strength",
        "entering_strength_rank"
    ]
].rename(
    columns={
        "team": "opponent",
        "entering_team_strength": "opponent_entering_strength",
        "entering_strength_rank": "opponent_entering_rank"
    }
)

team_games = team_games.merge(
    opponent_strength,
    on="opponent",
    how="left",
    validate="many_to_one"
)

display(
    team_games[
        [
            "team",
            "entering_strength_rank",
            "entering_team_strength",
            "opponent",
            "opponent_entering_rank",
            "opponent_entering_strength",
            "points_for",
            "points_against",
            "point_margin"
        ]
    ].sort_values(
        "entering_strength_rank"
    )
)

,team,entering_strength_rank,entering_team_strength,opponent,opponent_entering_rank,opponent_entering_strength,points_for,points_against,point_margin
3,BUF,1,7.483841,NYJ,31,-6.315289,31.0,12.0,19.0
12,JAX,2,5.875962,TEN,32,-6.584857,40.0,23.0,17.0
25,SF,3,4.990056,DAL,17,-0.453468,46.0,40.0,6.0
14,LA,4,4.643452,ARI,29,-4.795138,34.0,6.0,28.0
24,SEA,5,4.207138,LV,27,-4.330605,21.0,13.0,8.0
13,KC,7,3.818115,ATL,22,-2.831625,35.0,13.0,22.0
8,DET,8,3.609975,NE,15,1.490937,35.0,10.0,25.0
9,GB,10,3.446027,MIN,14,1.673649,10.0,30.0,-20.0
5,CIN,11,2.995699,PIT,12,2.844367,38.0,18.0,20.0
23,PIT,12,2.844367,CIN,11,2.995699,18.0,38.0,-20.0


# Expected Margin and Opponent-Adjusted Performance

The weekly update is based on how a team performed relative to expectation.

For each team game:

`performance residual = actual margin - expected margin`

Expected margin uses the difference between the two teams' entering strength ratings plus home field advantage.

This means beating a strong opponent is more informative than producing the same result against a weak opponent.

In [8]:
HOME_FIELD_POINTS = 1.5

team_games["expected_margin"] = (
    team_games["entering_team_strength"]
    - team_games["opponent_entering_strength"]
    + np.where(
        team_games["is_home"] == 1,
        HOME_FIELD_POINTS,
        -HOME_FIELD_POINTS
    )
)

team_games["performance_residual"] = (
    team_games["point_margin"]
    - team_games["expected_margin"]
)

display(
    team_games[
        [
            "team",
            "opponent",
            "entering_team_strength",
            "opponent_entering_strength",
            "point_margin",
            "expected_margin",
            "performance_residual"
        ]
    ].sort_values(
        "performance_residual",
        ascending=False
    )
)

,team,opponent,entering_team_strength,opponent_entering_strength,point_margin,expected_margin,performance_residual
18,MIN,GB,1.673649,3.446027,20.0,-3.272378,23.272378
8,DET,NE,3.609975,1.490937,25.0,3.619038,21.380962
14,LA,ARI,4.643452,-4.795138,28.0,7.938590,20.061410
5,CIN,PIT,2.995699,2.844367,20.0,1.651332,18.348668
13,KC,ATL,3.818115,-2.831625,22.0,5.149740,16.850260
6,CLE,HOU,-4.106187,2.325586,3.0,-4.931773,7.931773
15,LAC,BAL,-4.739916,-1.682630,3.0,-4.557286,7.557286
3,BUF,NYJ,7.483841,-6.315289,19.0,12.299130,6.700870
12,JAX,TEN,5.875962,-6.584857,17.0,10.960818,6.039182
4,CAR,NO,-3.664380,-2.781537,3.0,-2.382843,5.382843


# Control Extreme Single-Game Results

NFL game margins contain substantial single-game noise.

A blowout should affect team strength, but one extreme result should not completely redefine the team's rating.

Performance residuals are therefore capped before the weekly learning rate is applied.

In [9]:
RESIDUAL_SCALE = 14.0
MAX_WEEKLY_UPDATE = 1.5

team_games["weekly_strength_update"] = (
    MAX_WEEKLY_UPDATE
    * np.tanh(
        team_games["performance_residual"]
        / RESIDUAL_SCALE
    )
)

display(
    team_games[
        [
            "team",
            "opponent",
            "point_margin",
            "expected_margin",
            "performance_residual",
            "weekly_strength_update"
        ]
    ].sort_values(
        "weekly_strength_update",
        ascending=False
    )
)

,team,opponent,point_margin,expected_margin,performance_residual,weekly_strength_update
18,MIN,GB,20.0,-3.272378,23.272378,1.395792
8,DET,NE,25.0,3.619038,21.380962,1.364919
14,LA,ARI,28.0,7.938590,20.061410,1.338407
5,CIN,PIT,20.0,1.651332,18.348668,1.296648
13,KC,ATL,22.0,5.149740,16.850260,1.252120
6,CLE,HOU,3.0,-4.931773,7.931773,0.769237
15,LAC,BAL,3.0,-4.557286,7.557286,0.739262
3,BUF,NYJ,19.0,12.299130,6.700870,0.667723
12,JAX,TEN,17.0,10.960818,6.039182,0.609699
4,CAR,NO,3.0,-2.382843,5.382843,0.549899


# Offensive and Defensive Weekly Signals

Overall team strength uses the opponent adjusted margin residual.

We also preserve separate offensive and defensive signals because the weekly game projection model needs scoring information.

These are weekly updates, not cumulative season averages.

Opponent entering strength is used conservatively as context until separate entering offensive and defensive strength ratings are maintained.

In [10]:
league_points_per_team_game = (
    team_games["points_for"].mean()
)

OPPONENT_CONTEXT_RATE = 0.25
OFFENSE_DEFENSE_UPDATE_RATE = 0.15

team_games["raw_offense_performance"] = (
    team_games["points_for"]
    - league_points_per_team_game
)

team_games["raw_defense_performance"] = (
    league_points_per_team_game
    - team_games["points_against"]
)

team_games["adjusted_offense_performance"] = (
    team_games["raw_offense_performance"]
    + (
        OPPONENT_CONTEXT_RATE
        * team_games["opponent_entering_strength"]
    )
)

team_games["adjusted_defense_performance"] = (
    team_games["raw_defense_performance"]
    + (
        OPPONENT_CONTEXT_RATE
        * team_games["opponent_entering_strength"]
    )
)

team_games["weekly_offense_adjustment"] = (
    team_games["adjusted_offense_performance"]
    * OFFENSE_DEFENSE_UPDATE_RATE
)

team_games["weekly_defense_adjustment"] = (
    team_games["adjusted_defense_performance"]
    * OFFENSE_DEFENSE_UPDATE_RATE
)

display(
    team_games[
        [
            "team",
            "opponent",
            "points_for",
            "points_against",
            "opponent_entering_strength",
            "raw_offense_performance",
            "adjusted_offense_performance",
            "weekly_offense_adjustment",
            "raw_defense_performance",
            "adjusted_defense_performance",
            "weekly_defense_adjustment"
        ]
    ].sort_values(
        "team"
    )
)

,team,opponent,points_for,points_against,opponent_entering_strength,raw_offense_performance,adjusted_offense_performance,weekly_offense_adjustment,raw_defense_performance,adjusted_defense_performance,weekly_defense_adjustment
0,ARI,LA,6.0,34.0,4.643452,-18.178571,-17.017708,-2.552656,-9.821429,-8.660566,-1.299085
1,ATL,KC,13.0,35.0,3.818115,-11.178571,-10.224043,-1.533606,-10.821429,-9.866900,-1.480035
2,BAL,LAC,20.0,23.0,-4.739916,-4.178571,-5.363550,-0.804533,1.178571,-0.006408,-0.000961
3,BUF,NYJ,31.0,12.0,-6.315289,6.821429,5.242606,0.786391,12.178571,10.599749,1.589962
4,CAR,NO,20.0,17.0,-2.781537,-4.178571,-4.873956,-0.731093,7.178571,6.483187,0.972478
5,CIN,PIT,38.0,18.0,2.844367,13.821429,14.532520,2.179878,6.178571,6.889663,1.033449
6,CLE,HOU,27.0,24.0,2.325586,2.821429,3.402825,0.510424,0.178571,0.759968,0.113995
7,DAL,SF,40.0,46.0,4.990056,15.821429,17.068943,2.560341,-21.821429,-20.573915,-3.086087
8,DET,NE,35.0,10.0,1.490937,10.821429,11.194163,1.679124,14.178571,14.551306,2.182696
9,GB,MIN,10.0,30.0,1.673649,-14.178571,-13.760159,-2.064024,-5.821429,-5.403016,-0.810452


# Build Week-Level Feature Table

The final feature table contains only the new evidence that should move the prior week's ratings.

For Week 3:

`Week 3 strength = Week 2 strength + Week 2 performance update`

The previous week's rating therefore acts as the prior, and older games are not counted again.

In [11]:
team_features = (
    team_games
    .groupby("team", as_index=False)
    .agg(
        games_in_update=("game_id", "count"),

        entering_team_strength=(
            "entering_team_strength",
            "first"
        ),

        entering_strength_rank=(
            "entering_strength_rank",
            "first"
        ),

        average_opponent_strength=(
            "opponent_entering_strength",
            "mean"
        ),

        average_opponent_rank=(
            "opponent_entering_rank",
            "mean"
        ),

        actual_margin=(
            "point_margin",
            "mean"
        ),

        expected_margin=(
            "expected_margin",
            "mean"
        ),

        performance_residual=(
            "performance_residual",
            "mean"
        ),

        weekly_strength_update=(
            "weekly_strength_update",
            "mean"
        ),

        weekly_offense_adjustment=(
            "weekly_offense_adjustment",
            "mean"
        ),

        weekly_defense_adjustment=(
            "weekly_defense_adjustment",
            "mean"
        )
    )
)

display(
    team_features[
        [
            "team",
            "entering_strength_rank",
            "entering_team_strength",
            "average_opponent_rank",
            "average_opponent_strength",
            "actual_margin",
            "expected_margin",
            "performance_residual",
            "weekly_strength_update",
            "weekly_offense_adjustment",
            "weekly_defense_adjustment"
        ]
    ].sort_values(
        "weekly_strength_update",
        ascending=False
    )
)

,team,entering_strength_rank,entering_team_strength,average_opponent_rank,average_opponent_strength,actual_margin,expected_margin,performance_residual,weekly_strength_update,weekly_offense_adjustment,weekly_defense_adjustment
18,MIN,14,1.673649,10.0,3.446027,20.0,-3.272378,23.272378,1.395792,1.002440,2.256012
8,DET,8,3.609975,15.0,1.490937,25.0,3.619038,21.380962,1.364919,1.679124,2.182696
14,LA,4,4.643452,29.0,-4.795138,28.0,7.938590,20.061410,1.338407,1.293397,2.546968
5,CIN,11,2.995699,12.0,2.844367,20.0,1.651332,18.348668,1.296648,2.179878,1.033449
13,KC,7,3.818115,22.0,-2.831625,22.0,5.149740,16.850260,1.252120,1.517028,1.570600
6,CLE,25,-4.106187,13.0,2.325586,3.0,-4.931773,7.931773,0.769237,0.510424,0.113995
15,LAC,28,-4.739916,19.0,-1.682630,3.0,-4.557286,7.557286,0.739262,-0.239884,0.563687
3,BUF,1,7.483841,31.0,-6.315289,19.0,12.299130,6.700870,0.667723,0.786391,1.589962
12,JAX,2,5.875962,32.0,-6.584857,17.0,10.960818,6.039182,0.609699,2.126282,-0.070146
4,CAR,24,-3.664380,21.0,-2.781537,3.0,-2.382843,5.382843,0.549899,-0.731093,0.972478


# Sanity Checks

Before saving, verify that:

- all 32 teams are represented
- every team has an entering Week 2 rating
- every opponent has an entering Week 2 rating
- no Week 3 game results were used
- weekly strength updates remain within the intended cap

In [12]:
print("Teams in feature table:", len(team_features))
print(
    "Unique teams:",
    team_features["team"].nunique()
)

print(
    "Missing entering team strengths:",
    team_features[
        "entering_team_strength"
    ].isna().sum()
)

print(
    "Missing opponent strengths:",
    team_features[
        "average_opponent_strength"
    ].isna().sum()
)

print(
    "Largest positive weekly update:",
    round(
        team_features[
            "weekly_strength_update"
        ].max(),
        3
    )
)

print(
    "Largest negative weekly update:",
    round(
        team_features[
            "weekly_strength_update"
        ].min(),
        3
    )
)

if (
    completed_games["week"] >= TARGET_WEEK
).any():
    raise ValueError(
        "DATA LEAKAGE: target-week results "
        "were found in completed games."
    )

if len(team_features) != 32:
    print()
    print(
        "WARNING: Feature table does not "
        "contain all 32 teams."
    )

Teams in feature table: 28
Unique teams: 28
Missing entering team strengths: 0
Missing opponent strengths: 0
Largest positive weekly update: 1.396
Largest negative weekly update: -1.396



# Save In Season Feature Table

This output will be consumed by:

`04_Weekly_Team_Strength.ipynb`

The file contains only the new in season evidence. It does not overwrite or replace the preseason team strength table.


In [13]:
output_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_inseason_features.parquet"
)

team_features = team_features.sort_values(
    "team"
).reset_index(drop=True)

team_features.to_parquet(
    output_path,
    index=False
)

print("Saved:", output_path)


Saved: ..\..\data\processed\weekly\week_11_inseason_features.parquet


# Interpretation

The weekly model is now sequential.

For Week 3:

1. Week 2 team strength is treated as the entering prior.
2. Only Week 2 game results provide new evidence.
3. Performance is evaluated relative to the opponent's entering strength.
4. Extreme single game residuals are capped.
5. A conservative learning rate determines the weekly rating movement.
6. The resulting update will be added to the Week 2 rating in `04_Weekly_Team_Strength.ipynb`.

This prevents older games from being counted repeatedly and makes strength of opponent part of every weekly update.

The frozen preseason model remains unchanged.